# Vocalyze — AI Audio Intelligence Assistant

**Samsung Campus AI · Group 25**

A meeting recording becomes a **speaker-attributed transcript** and a brief in
which **every point carries the line it came from**.

| # | Section | Owner | What it establishes |
|---|---|---|---|
| 1 | Data & Benchmarking | Turki | AMI and ICSI loaded, speaker-disjoint splits, 16 kHz mono |
| 2 | ASR & Performance | Taghreed | Whisper WER, accuracy and latency |
| 3 | Speaker Diarization | Reema | pyannote, DER on ICSI |
| 4 | Grounded Summarisation | Aljawharah | LLM brief over the attributed transcript |
| 5 | Integration & Privacy | Naif | The service that merges all of it, and the controls over it |
| 6 | **Run the whole system** | — | A live site, driven by this notebook |

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Sections 1–4 build and measure the components. Section 5 is the layer that
turns them into one product. Section 6 runs that product on a public link.

---

## 0 · Environment

In [ ]:
import torch

print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

---

# 1 · Data & Benchmarking
### Turki

Two multi-speaker meeting corpora, acquired programmatically. The split is
**speaker-disjoint**: a model that has heard a speaker in training scores
better on that speaker at test time, and the number stops meaning anything.

In [ ]:
# Install libraries required to access and analyse the AMI dataset
2
# datasets is used to load datasets from Hugging Face
3
# pandas will be used later for data analysis and dataset splitting
!pip install datasets huggingface_hub pandas

In [ ]:
from datasets import get_dataset_config_names

configs = get_dataset_config_names("edinburghcstr/ami")
print(configs)

In [ ]:
from datasets import load_dataset

metadata = load_dataset(
    "edinburghcstr/ami",
    "ihm",
    split="train",
    streaming=False
).remove_columns(["audio"])

In [ ]:
unique_speakers = set()

for sample in metadata:
    unique_speakers.add(sample["speaker_id"])

print("Unique Speakers:", len(unique_speakers))

In [ ]:
# Calculate speaker-based split sizes

total_speakers = len(unique_speakers)

train_speakers = round(total_speakers * 0.70)
validation_speakers = round(total_speakers * 0.15)

test_speakers = total_speakers - train_speakers - validation_speakers

print(f"Total Speakers: {total_speakers}")
print(f"Train Speakers (70%): {train_speakers}")
print(f"Validation Speakers (15%): {validation_speakers}")
print(f"Test Speakers (15%): {test_speakers}")

In [ ]:
# Convert unique speakers into a list
# Shuffle speakers randomly before splitting
# This helps create unbiased train/validation/test sets

import random

speaker_list = list(unique_speakers)

random.seed(42)

random.shuffle(speaker_list)

In [ ]:
# Create speaker-disjoint train, validation, and test speaker groups
# Each speaker will belong to only one dataset split

train_speaker_ids = speaker_list[:109]

validation_speaker_ids = speaker_list[109:132]

test_speaker_ids = speaker_list[132:]

print("Train Speakers:", len(train_speaker_ids))
print("Validation Speakers:", len(validation_speaker_ids))
print("Test Speakers:", len(test_speaker_ids))

In [ ]:
# Verify that no speaker appears in more than one split
# This confirms that Speaker-Disjoint Split was successfully applied

print("Train ∩ Validation =", len(set(train_speaker_ids) & set(validation_speaker_ids)))

print("Train ∩ Test =", len(set(train_speaker_ids) & set(test_speaker_ids)))

print("Validation ∩ Test =", len(set(validation_speaker_ids) & set(test_speaker_ids)))

In [ ]:
from datasets import load_dataset, Audio

# نعيد تحميل الداتاست، هذه المرة بدون حذف عمود audio
# لأن هذه المرحلة تحتاج الصوت الفعلي للمعالجة
audio_dataset = load_dataset(
    "edinburghcstr/ami",
    "ihm",
    split="train",
    streaming=True
)

# تحويل معدل العينات إلى 16kHz (lazy — يصير فقط وقت سحب العينة فعليًا)
audio_dataset = audio_dataset.cast_column("audio", Audio(sampling_rate=16000))

# نتحقق من أول عينة: معدل العينات + شكل القناة الصوتية
sample = next(iter(audio_dataset))
print("Sampling rate:", sample["audio"]["sampling_rate"])
print("Array shape:", sample["audio"]["array"].shape)

In [ ]:
# Load one sample from the ICSI dataset
# The dataset only provides a test split

from datasets import load_dataset

icsi_dataset = load_dataset(
    "argmaxinc/icsi-meetings",
    split="test",
    streaming=True
)

sample = next(iter(icsi_dataset))

print(sample)

In [ ]:
# Inspect ICSI dataset structure
# This helps identify available fields for benchmarking and evaluation

print(sample.keys())

In [ ]:
# ==========================================
# Part 1 (updated): validate a single sample
# ==========================================

def audio_info(sample):
    """Return (duration, sample_rate, channels) for either audio shape.

    A streaming dataset gives a decoder object with .metadata; a downloaded one
    gives {"array", "sampling_rate"}. Section 3 uses the downloaded form, so a
    validator that reads only one shape rejects every sample there and the DER
    loop reports zero meetings.
    """
    audio = sample["audio"]
    if isinstance(audio, dict):
        array, rate = audio["array"], audio["sampling_rate"]
        channels = 1 if getattr(array, "ndim", 1) == 1 else array.shape[0]
        return len(array) / rate, rate, channels
    meta = audio.metadata
    return meta.duration_seconds, meta.sample_rate, meta.num_channels


def is_valid_sample(sample):
    # Check audio metadata is accessible and has a valid duration
    try:
        duration, _rate, _channels = audio_info(sample)
        if duration <= 0:
            return False
    except Exception:
        return False

    # Check speaker labels exist
    if len(sample["speakers"]) == 0:
        return False

    # Check timestamps exist
    if len(sample["timestamps_start"]) == 0 or len(sample["timestamps_end"]) == 0:
        return False

    # Check that speakers / start / end lists are the same length
    if not (len(sample["speakers"]) == len(sample["timestamps_start"]) == len(sample["timestamps_end"])):
        return False

    return True


# ==========================================
# Part 1b (new): check 16kHz + Mono compliance across the whole dataset
# ==========================================

CHECK_LIMIT = 20        # bounded: this streams meeting-length audio

non_compliant = 0
checked = 0

for sample in icsi_dataset:
    if checked >= CHECK_LIMIT:
        break
    checked += 1
    _duration, rate, channels = audio_info(sample)
    if rate != 16000 or channels != 1:
        non_compliant += 1

print(f"Checked: {checked}")
print(f"Not already 16kHz Mono: {non_compliant}")

---

# 2 · ASR & Performance
### Taghreed

**Target: WER ≤ 10%.** Text is normalised before comparison — punctuation and
case would otherwise make an identical transcription look like errors — and
one- and two-word utterances are excluded, since a single substitution there
is a WER of 1.0.

In [ ]:
!pip install -q openai-whisper
!pip install -q openai-whisper jiwer librosa datasets

In [ ]:
# ============================================================
# Whisper ASR + Baseline + WER Evaluation + Performance
# Results are displayed as percentages
# ============================================================

# Install first if needed:
# pip install openai-whisper jiwer torch

import whisper
import time
from jiwer import wer


# ------------------------------------------------------------
# 1. Load Whisper ASR Model
# ------------------------------------------------------------
model = whisper.load_model("base")

# Audio file
audio_file = "audio.wav"

# ------------------------------------------------------------
# 2. Whisper ASR - Transcription
# ------------------------------------------------------------
start_time = time.time()

result = model.transcribe(audio_file)
predicted_text = result["text"].strip()

end_time = time.time()

inference_time = end_time - start_time

print("\n========== Whisper ASR ==========")
print("Predicted Text:")
print(predicted_text)


# ------------------------------------------------------------
# 3. Baseline / Ground Truth
# ------------------------------------------------------------
# ضع النص الصحيح (Ground Truth) هنا
ground_truth = """
اكتب هنا النص الصحيح الموجود في التسجيل الصوتي
""".strip()


# ------------------------------------------------------------
# 4. WER Evaluation
# ------------------------------------------------------------
word_error_rate = wer(ground_truth, predicted_text)

# Convert WER to percentage
wer_percentage = word_error_rate * 100

# Accuracy = 100% - WER
accuracy_percentage = max(0, 100 - wer_percentage)


# ------------------------------------------------------------
# 5. Performance Measurement
# ------------------------------------------------------------

# Number of words in Ground Truth
reference_words = len(ground_truth.split())

# Number of words predicted by Whisper
predicted_words = len(predicted_text.split())

# Real-Time Factor (RTF)
# Lower is better
# If audio duration is available from Whisper
audio_duration = result.get("segments", [])

if audio_duration:
    audio_length = audio_duration[-1]["end"]
    rtf = inference_time / audio_length if audio_length > 0 else 0
else:
    audio_length = 0
    rtf = 0


# ------------------------------------------------------------
# 6. Display Results as Percentages
# ------------------------------------------------------------

print("\n========== Evaluation Results ==========")

print(f"WER: {wer_percentage:.2f}%")
print(f"ASR Accuracy: {accuracy_percentage:.2f}%")

print("\n========== Performance ==========")

print(f"Inference Time: {inference_time:.2f} seconds")

if audio_length > 0:
    print(f"Audio Duration: {audio_length:.2f} seconds")
    print(f"Real-Time Factor: {rtf:.2f}")

print(f"Reference Words: {reference_words}")
print(f"Predicted Words: {predicted_words}")


# ------------------------------------------------------------
# 7. Simple Performance Score (%)
# ------------------------------------------------------------
# Here we use ASR accuracy as the main performance percentage.

performance_percentage = accuracy_percentage

print("\n========== Final Performance ==========")
print(f"Overall Performance: {performance_percentage:.2f}%")

In [ ]:
from datasets import load_dataset
import whisper
import librosa
from jiwer import wer
import time, re

# ================================
# دالة تطبيع النص قبل المقارنة
# ================================
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ================================
# دالة لاستبعاد العينات القصيرة/غير الصالحة
# ================================
def is_degenerate_sample(ref_text, min_words=3):
    return len(ref_text.split()) < min_words

# ================================
# تحميل داتا AMI + تحميل الموديل مرة واحدة فقط
# ================================
ami = load_dataset("edinburghcstr/ami", "ihm", split="test")
whisper_model = whisper.load_model("small")

def whisper_asr(audio_array, sample_rate):
    if sample_rate != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=16000)
    result = whisper_model.transcribe(audio_array, language="en")
    return result["text"]

def measure_performance(audio_array, sample_rate):
    if sample_rate != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=16000)
    start = time.time()
    result = whisper_model.transcribe(audio_array, language="en")
    end = time.time()
    return end - start, result["text"]

# ================================
# التقييم على عدة عينات (مو عينة وحدة)
# ================================
wer_scores = []
latencies = []
skipped = 0

for i, sample in enumerate(ami.select(range(100))):
    ref_raw = sample["text"]

    if is_degenerate_sample(ref_raw):
        skipped += 1
        continue

    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    pred_raw = whisper_asr(audio, sr)
    ref = normalize_text(ref_raw)
    pred = normalize_text(pred_raw)

    sample_wer = wer(ref, pred)
    wer_scores.append(sample_wer)

    latency, _ = measure_performance(audio, sr)
    latencies.append(latency)

    print(f"[{i}] REF: {ref[:50]!r} | PRED: {pred[:50]!r} | WER: {sample_wer:.2f}")

avg_wer = sum(wer_scores) / len(wer_scores) if wer_scores else None
avg_latency = sum(latencies) / len(latencies) if latencies else None

print(f"\nتم تجاهل {skipped} عينة قصيرة/غير صالحة")
print(f"متوسط WER عبر {len(wer_scores)} عينة: {avg_wer:.3f}" if avg_wer else "لا عينات صالحة")
print(f"متوسط زمن الاستجابة: {avg_latency:.3f} ثانية" if avg_latency else "")

In [ ]:
# ================================
# حساب Accuracy بشكل صحيح (بدون أرقام سالبة غريبة)
# ================================

if avg_wer is not None:
    # نمنع القيمة السالبة إذا WER تجاوز 100% لأي عينة شاذة أفلتت من الفلترة
    accuracy_percent = max(0, (1 - avg_wer) * 100)
    print(f"Whisper Accuracy (%): {accuracy_percent:.2f}")
else:
    print("لا توجد عينات كافية لحساب Accuracy")

In [ ]:
from datasets import load_dataset
import time

def load_icsi_dataset(max_retries=5, wait_seconds=2):
    """
    تحميل ICSI مع إعادة محاولات تلقائية في حال ظهور خطأ 503.
    """
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Loading ICSI dataset... Attempt {attempt}/{max_retries}")
            dataset = load_dataset("argmaxinc/icsi-meetings", split="test")
            print("ICSI dataset loaded successfully.")
            return dataset

        except Exception as e:
            print(f"Error: {e}")
            print(f"Retrying in {wait_seconds} seconds...")
            time.sleep(wait_seconds)

    print("Failed to load ICSI dataset after multiple attempts.")
    return None

# تحميل الداتا
icsi_dataset = load_icsi_dataset()

if icsi_dataset is None:
    print("Dataset could not be loaded. Please try again later.")
else:
    print("Dataset is ready for processing.")

In [ ]:
!pip install git+https://github.com/openai/whisper.git
!pip install librosa

import whisper
import librosa

# تحميل نموذج Whisper
model = whisper.load_model("base")

In [ ]:
import whisper
import librosa

# تحميل نموذج Whisper (Baseline لاحقاً)
model = whisper.load_model("base")

def whisper_asr(audio_array, sample_rate):
    """
    تشغيل Whisper ASR على عينة صوتية واحدة.
    """
    # Whisper يعمل على 16kHz فقط
    if sample_rate != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=16000)

    result = model.transcribe(audio_array)
    return result["text"]

In [ ]:
# استخراج أول عينة من ICSI
sample = next(iter(icsi_dataset))

# التأكد أن العينة تحتوي صوت
if "audio" not in sample or sample["audio"] is None:
    print("This sample has no audio.")
else:
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

In [ ]:
pred = whisper_asr(audio, sr)
print(pred)

In [ ]:
# اختيار نموذج Whisper الأساسي (Baseline)
baseline_model = whisper.load_model("base")

def baseline_asr(audio_array, sample_rate):
    """
    تشغيل نموذج Whisper الأساسي (Baseline).
    """
    if sample_rate != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=16000)

    result = baseline_model.transcribe(audio_array)
    return result["text"]

In [ ]:
baseline_pred = baseline_asr(audio, sr)
print(baseline_pred)

In [ ]:
from jiwer import wer

def compute_wer(reference_text, predicted_text):
    """
    حساب معدل الخطأ في الكلمات WER.
    """
    references = [reference_text]
    predictions = [predicted_text]

    if len(references) == 0 or len(predictions) == 0:
        print("No valid samples to evaluate WER.")
        return None

    score = wer(references, predictions)
    return score

In [ ]:
from datasets import load_dataset

ami = load_dataset("edinburghcstr/ami", "ihm", split="test")

sample = ami[0]

audio = sample["audio"]["array"]
sr = sample["audio"]["sampling_rate"]
ref = sample["text"]   # النص موجود هنا

pred = whisper_asr(audio, sr)

wer_score = compute_wer(ref, pred)
print("WER:", wer_score)

In [ ]:
from datasets import load_dataset

# تحميل داتا AMI
ami = load_dataset("edinburghcstr/ami", "ihm", split="test")

# أخذ أول عينة
sample = ami[0]

# استخراج النص فقط
ref = sample["text"]

print("Transcript:", ref)

In [ ]:
from datasets import load_dataset

ami = load_dataset("edinburghcstr/ami", "ihm", split="test")

sample = ami[0]
audio = sample["audio"]["array"]
sr = sample["audio"]["sampling_rate"]
ref = sample["text"]   # هنا النص الحقيقي

pred = whisper_asr(audio, sr)
wer_score = compute_wer(ref, pred)

print("Whisper Output:", pred)
print("WER:", wer_score)

In [ ]:
# استخراج أول عينة من ICSI
sample = next(iter(icsi_dataset))

# استخراج الصوت فقط (لأن ICSI ما فيها نصوص)
audio = sample["audio"]["array"]
sr = sample["audio"]["sampling_rate"]

# تشغيل Whisper على الصوت
pred = whisper_asr(audio, sr)
print("Whisper Output:", pred)

In [ ]:
sample = next(iter(icsi_dataset))

audio = sample["audio"]["array"]
sr = sample["audio"]["sampling_rate"]

pred = whisper_asr(audio, sr)
print("Whisper Output:", pred)

In [ ]:
# استخراج أول عينة بشكل آمن
sample = next(iter(icsi_dataset))

# التأكد أن العينة تحتوي نص
if "transcript" not in sample or sample["transcript"] is None:
    print("This sample has no transcript.")
else:
    ref = sample["transcript"]          # النص الصحيح
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    pred = whisper_asr(audio, sr)       # تشغيل Whisper

    # الآن نحسب WER بدون خطأ
    wer_score = compute_wer(ref, pred)
    print("WER:", wer_score)

In [ ]:
wer_score = compute_wer(ref, pred)
print("WER:", wer_score)

In [ ]:
import time

def measure_performance(audio_array, sample_rate):
    """
    قياس زمن المعالجة (Latency) ووقت التشغيل.
    """
    if sample_rate != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=16000)

    start = time.time()
    result = model.transcribe(audio_array)
    end = time.time()

    latency = end - start
    return latency, result["text"]

In [ ]:
latency, output = measure_performance(audio, sr)
print("Latency:", latency)
print("Output:", output)

---

# 3 · Speaker Diarization
### Reema

**Target: DER ≤ 12%.** Diarization answers *who spoke when*, scored as missed
speech plus false alarm plus speaker confusion over total reference speech.

`HF_TOKEN` goes in Colab under 🔑 → *Add new secret*, and the terms must be
accepted with that same account on
[speaker-diarization-3.1](https://hf.co/pyannote/speaker-diarization-3.1) and
[segmentation-3.0](https://hf.co/pyannote/segmentation-3.0).

In [ ]:
!pip install pyannote.audio torch soundfile

import torch
import numpy as np
from pyannote.audio import Pipeline
from pyannote.core import Segment, Annotation
from pyannote.metrics.diarization import DiarizationErrorRate

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading Diarization Pipeline on: {device}")

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN
)
pipeline.to(device)

In [ ]:
def build_reference_annotation(sample):
   #Convert reference speaker IDs and start/end timestamps into a pyannote Annotation object.
    ref_annotation = Annotation(uri="meeting_audio")
    speakers = sample.get("speakers", [])
    starts = sample.get("timestamps_start", [])
    ends = sample.get("timestamps_end", [])

    for spk, start, end in zip(speakers, starts, ends):
        if end > start:
            ref_annotation[Segment(start, end)] = str(spk)

    return ref_annotation

In [ ]:
def run_diarization_on_sample(audio_array, sample_rate):

   # Send the audio waveform array to the PyAnnote pipeline and extract speaker segments.
    if isinstance(audio_array, np.ndarray):
        audio_tensor = torch.from_numpy(audio_array).float()
    else:
        audio_tensor = audio_array.float()

    if audio_tensor.ndim == 1:
        audio_tensor = audio_tensor.unsqueeze(0)

    waveform = {"waveform": audio_tensor, "sample_rate": sample_rate}

    # Run speaker diarization inference
    result = pipeline(waveform)

    # If pyannote returns a DiarizeOutput object, extract the Annotation
    if hasattr(result, "speaker_diarization"):
        return result.speaker_diarization

    return result

In [ ]:
#Run DER evaluation across the dataset samples

der_metric = DiarizationErrorRate()
processed_count = 0

print("\nStarting Diarization & DER Evaluation...")
print("=" * 60)

for sample in icsi_dataset:
    if not is_valid_sample(sample):
        continue

    processed_count += 1

    audio_arr = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    # Construct Ground Truth annotation
    ref_annot = build_reference_annotation(sample)

    # Run model hypothesis
    hyp_annot = run_diarization_on_sample(audio_arr, sr)

    # Compute DER for the current sample and accumulate in metric
    sample_der = der_metric(ref_annot, hyp_annot)

    print(f"Sample {processed_count}:")
    print(f"  Ground Truth Segments: {len(ref_annot)}")
    print(f"  Predicted Segments:    {len(hyp_annot)}")
    print(f"  Sample DER:            {sample_der * 100:.2f}%")

    # Display the first three detected speaker turns for inspection
    print("  First Predicted Turns:")
    for turn, _, speaker in list(hyp_annot.itertracks(yield_label=True))[:3]:
        print(f"    - [{turn.start:.2f}s -> {turn.end:.2f}s] Speaker {speaker}")
    print("-" * 60)

    # Limit to 5 samples for the initial benchmark run to control GPU runtime
    if processed_count >= 5:
        break

In [ ]:
final_der = abs(der_metric)
print("\n" + "=" * 40)
print(f"Total Meetings Evaluated: {processed_count}")
print(f"Overall Cumulative DER:   {final_der * 100:.2f}%")
print("=" * 40)

---

# 4 · Grounded Summarisation
### Aljawharah

The transcript carries speakers; this turns it into a brief. Attribution is
**overlap-maximising** rather than boundary-matching: each line goes to the
speaker whose turns cover most of it, with an explicit `UNKNOWN` instead of a
guess.

In [ ]:
#  Load LLM

# Imported under its own name: section 3 binds `pipeline` to the pyannote
# object, so the bare name here would call the diarizer instead.
from transformers import pipeline as hf_pipeline

device = 0 if torch.cuda.is_available() else -1

llm = hf_pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=device
)

print("LLM loaded successfully.")

In [ ]:
# ============================================================
# Cell 3: Create Speaker-Attributed Transcript
# ============================================================

def create_speaker_transcript(whisper_result, diarization):
    """
    Combine Whisper transcription segments with
    PyAnnote speaker information.
    """

    speaker_transcript = []

    whisper_segments = whisper_result.get("segments", [])

    for segment in whisper_segments:

        start = segment["start"]
        end = segment["end"]
        text = segment["text"].strip()

        if not text:
            continue

        best_speaker = "UNKNOWN"
        best_overlap = 0

        for turn, _, speaker in diarization.itertracks(
            yield_label=True
        ):

            overlap_start = max(start, turn.start)
            overlap_end = min(end, turn.end)

            overlap = max(
                0,
                overlap_end - overlap_start
            )

            if overlap > best_overlap:
                best_overlap = overlap
                best_speaker = speaker

        speaker_transcript.append(
            f"[{start:.2f}s - {end:.2f}s] "
            f"{best_speaker}: {text}"
        )

    return "\n".join(speaker_transcript)

---

# 5 · Integration & Privacy
### Naif

The layer that turns three components into one system: the interface, the
upload flow, the FastAPI service, the merge of ASR + diarization + summary,
and the privacy controls over all of it.

| Requirement | Where it lives |
|---|---|
| UI | `app/web/` — no build step, no framework |
| Upload flow | `POST /v1/jobs` — validate, accept, process behind a job id |
| Backend | `app/main.py` + `app/api/` |
| Integration | `app/pipeline/align_core.py` — every word to its speaker |
| Privacy | `crypto.py` · `store.py` · `audit.py` · `retention.py` · `redaction.py` |

**The architectural point.** Whisper answers *what was said and when*. pyannote
answers *who was speaking and when*. Neither answers **who said what** — and
that is the product. So it is written here, in the integration layer, rather
than inside either model. Then every point in the brief is verified against the
transcript before it is shown.

### 5.1 · The service, and its tests

The tests run against scripted backends holding the same contracts as the real
models: no weights, no network, the same answer on any machine.

In [ ]:
!git clone --depth 1 https://github.com/imzezsv-dot/vocalyze.git /content/vocalyze 2>/dev/null || echo "already cloned"
%cd /content/vocalyze
!pip install -q -r requirements-dev.txt
!pytest -q

### 5.2 · Upload → five stages → result

Upload is a two-step exchange, not one long request: the file is validated and
accepted, and the work happens behind a job id. A ninety-minute meeting takes
minutes to transcribe, and a POST held open for all of it dies at the first
proxy timeout.

In [ ]:
import json
import os
import sys
import time

sys.path.insert(0, "/content/vocalyze")

from app.core.crypto import generate_service_key

os.environ["ENCRYPTION_KEY"] = generate_service_key()   # never in the repo
os.environ["DATA_DIR"] = "/content/vocalyze-data"
os.environ["ASR_BACKEND"] = "mock"            # scripted, for the walkthrough
os.environ["DIARIZATION_BACKEND"] = "mock"
os.environ["SUMMARIZER_BACKEND"] = "mock"

from fastapi.testclient import TestClient

import app.main

client = TestClient(app.main.app)
client.__enter__()          # runs the worker and the retention sweeper

print(json.dumps(client.get("/v1/capabilities").json(), indent=2)[:400], "…")

In [ ]:
# A real audio file, enough to exercise decoding and 16 kHz normalisation.
!ffmpeg -y -f lavfi -i "sine=frequency=220:duration=12" -ar 44100 -ac 2 /content/meeting.wav -loglevel error

# Consent is checked before the bytes are touched.
refused = client.post(
    "/v1/jobs",
    files={"file": ("meeting.wav", open("/content/meeting.wav", "rb"), "audio/wav")},
    data={"consent": "false"},
)
print(refused.status_code, refused.json()["error"], "->", refused.json()["fix"])

accepted = client.post(
    "/v1/jobs",
    files={"file": ("meeting.wav", open("/content/meeting.wav", "rb"), "audio/wav")},
    data={"consent": "true"},
).json()

job_id, token = accepted["job_id"], accepted["access_token"]
headers = {"Authorization": f"Bearer {token}"}      # the id alone grants nothing

while True:
    job = client.get(f"/v1/jobs/{job_id}", headers=headers).json()
    if job["state"] in ("completed", "failed"):
        break
    time.sleep(0.3)

for stage in job["stages"]:
    print(f"  {stage['name']:<13} {stage['state']:<8} {stage.get('detail') or ''}")

### 5.3 · Who said what, and the brief that cites it

Every generated point is treated as a claim to be checked: it must cite lines
that exist, its wording must appear in them, and any figure it states must
appear in the evidence. What fails is dropped and counted.

In [ ]:
result = client.get(f"/v1/jobs/{job_id}/result", headers=headers).json()
transcript, brief, quality = result["transcript"], result["brief"], result["quality"]

print("Speakers:", transcript["speakers"])
print("=" * 70)
for utterance in transcript["utterances"][:6]:
    mark = "  [crosstalk]" if utterance["overlapped"] else ""
    print(f"[{utterance['id']:>3}] {utterance['speaker']:<12} {utterance['text'][:52]}{mark}")

print("\n" + "=" * 70)
print("Summary:", brief["summary"][:200])
print("=" * 70)
by_id = {u["id"]: u["text"] for u in transcript["utterances"]}
for label, key in [("DECISION", "decisions"), ("ACTION", "action_items")]:
    for item in brief[key]:
        print(f"• [{label}] {item['text'][:60]}")
        print(f"        cites {item['evidence']['utterance_ids']} "
              f"— match {item['evidence']['grounding']:.0%}")

print(f"\nVerified: {quality['grounded_claims']} | Dropped as unsupported: {quality['dropped_claims']}")

### 5.4 · Privacy, as assertions rather than promises

Enforced **between** the stages, because that is where the data moves: audio is
encrypted on arrival with a key for that job alone and destroyed as soon as it
has been read; identifiers are removed before storage and before anything
reaches the summariser; every action is written to a hash-chained trail that
never contains transcript text.

In [ ]:
from pathlib import Path

privacy = result["privacy"]
print("Audio still stored? ", privacy["audio_retained"])
print("Encrypted at rest?  ", privacy["encrypted_at_rest"])

# The bytes on disk really are unreadable.
blob = Path(os.environ["DATA_DIR"]) / "jobs" / job_id / "result.enc"
raw = blob.read_bytes()
print("\nDoes the meeting text appear inside the stored file?",
      transcript["utterances"][0]["text"].encode() in raw)

# Each audit line carries the hash of the one before it.
print("Audit chain:", client.get("/v1/privacy/audit/verify").json())

# Redaction is deliberately narrow — budgets and versions are meeting substance.
from app.pipeline.redaction import redact_text
for line in ["Email me at layla.ahmed@example.com or call +966 50 123 4567",
             "Error rate is 11.2 percent, the budget is 250000 riyals, we shipped 3.11.9"]:
    print("\nbefore:", line)
    print("after :", redact_text(line)[0])

# The right to erasure destroys the key, not just the file.
print("\n" + client.delete(f"/v1/jobs/{job_id}", headers=headers).json()["message"])
print("Read after deletion:", client.get(f"/v1/jobs/{job_id}", headers=headers).status_code)

client.__exit__(None, None, None)

### 5.5 · How the team's models plug in

The integration layer never imports Whisper or pyannote. It depends on three
contracts, so each owner can replace their implementation without a line
changing in the API, the aligner or the interface. Switching is configuration:

```ini
ASR_BACKEND=whisper           # section 2
DIARIZATION_BACKEND=pyannote  # section 3
SUMMARIZER_BACKEND=llm        # section 4
```

In [ ]:
import inspect

from app.pipeline.asr import ASRBackend
from app.pipeline.diarization import DiarizationBackend
from app.pipeline.summarizer import SummarizerBackend

for contract in (ASRBackend, DiarizationBackend, SummarizerBackend):
    print(inspect.getsource(contract))

---

# 6 · Run the whole system

**This is the demo.** The cell below runs the service here — real Whisper and
real pyannote on this GPU — opens a public tunnel to it, and displays the
published interface **already connected to this notebook**.

The site and the pipeline are not two separate things: the page is the front
end, this runtime is the back end. Upload a recording and every line comes back
with who said it, and a brief in which every point cites the line it came from.

Without `HF_TOKEN` the transcription is still real, but lines come back
unattributed rather than carrying a speaker name invented from a fixture.

In [ ]:
# ============================================================================
# The live system. Self-contained: run this cell on its own.
# ============================================================================
import os
import re
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

ROOT = Path("/content/vocalyze")
SITES = ["https://vocalyze-ai.github.io/", "https://imzezsv-dot.github.io/vocalyze/"]

if not (ROOT / "app").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/imzezsv-dot/vocalyze.git", str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "-q"], check=False)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

print("Installing…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", "requirements.txt", "faster-whisper"], check=True)
subprocess.run("apt-get -qq install -y ffmpeg > /dev/null", shell=True, check=False)

CLOUDFLARED = "/usr/local/bin/cloudflared"
if not Path(CLOUDFLARED).exists():
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        f"cloudflared-linux-amd64 -O {CLOUDFLARED} && chmod +x {CLOUDFLARED}",
        shell=True, check=True)

# ----------------------------------------------------------------- models
HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or ""
except Exception:
    pass

# "none", not the scripted diarizer: that one replays the sample meeting's
# turns, which over a real recording become speaker names invented from a
# fixture. Unattributed is the truthful answer when nothing knows.
diarization = "none"
if HF_TOKEN:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyannote.audio"], check=False)
    try:
        from huggingface_hub import model_info
        for repo in ("pyannote/speaker-diarization-3.1", "pyannote/segmentation-3.0"):
            model_info(repo, token=HF_TOKEN)
        diarization = "pyannote"
        print("pyannote available — real speaker separation.")
    except Exception as exc:
        print(f"pyannote not usable by this account ({exc.__class__.__name__}).")

if diarization != "pyannote":
    print("\n" + "!" * 72)
    print("  NO SPEAKER SEPARATION — set HF_TOKEN and accept the pyannote terms.")
    print("  Transcription is real; lines come back unattributed.")
    print("!" * 72)

os.environ.update(
    ASR_BACKEND="whisper", WHISPER_MODEL="small",
    DIARIZATION_BACKEND=diarization, SUMMARIZER_BACKEND="extractive",
    HUGGINGFACE_TOKEN=HF_TOKEN, DATA_DIR="/content/vocalyze-data",
    MAX_UPLOAD_MB="200",
    CORS_ORIGINS="https://vocalyze-ai.github.io,https://imzezsv-dot.github.io",
)
from app.core.crypto import generate_service_key
os.environ["ENCRYPTION_KEY"] = generate_service_key()

# ----------------------------------------------------------------- launch
log = open("/content/vocalyze-service.log", "w")
service = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT, cwd=str(ROOT))

print("Starting the service…")
for _ in range(90):
    if service.poll() is not None:
        print(Path("/content/vocalyze-service.log").read_text()[-1500:])
        raise SystemExit("The service failed to start.")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/health", timeout=2) as r:
            if r.status == 200:
                break
    except (urllib.error.URLError, OSError):
        time.sleep(1)
else:
    raise SystemExit("The service did not become healthy.")

# ----------------------------------------------------------------- tunnel
tunnel = subprocess.Popen([CLOUDFLARED, "tunnel", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1)
api = None
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        break
    found = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if found:
        api = found.group(0)
        break
if not api:
    raise SystemExit("Could not open the tunnel — re-run this cell.")

for _ in range(30):                     # wait for the tunnel to route
    try:
        with urllib.request.urlopen(f"{api}/v1/health", timeout=5) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(2)


def reads_backend(site):
    """Does this published build accept `?api=`? Its own script says so."""
    try:
        with urllib.request.urlopen(f"{site}app.js", timeout=10) as r:
            return b"externalApi" in r.read()
    except Exception:
        return False


site = next((s for s in SITES if reads_backend(s)), None)
if site is None:
    raise SystemExit("No published build can reach a backend. Re-publish the site.")

link = f"{site}?api={api}"

# ------------------------------------------------------------------ open it
from IPython.display import HTML, display

speakers = "pyannote (real)" if diarization == "pyannote" else "not available"
display(HTML(f"""
<div style="font-family:system-ui;border:1px solid #C9C0A8;background:#FBF7EB;
            padding:20px 24px;margin:12px 0;max-width:820px">
  <div style="font-size:11px;letter-spacing:.18em;text-transform:uppercase;
              color:#737870">Vocalyze &middot; live</div>
  <a href="{link}" target="_blank" rel="noopener"
     style="display:inline-block;margin-top:10px;font-size:24px;font-weight:700;
            color:#0F4A34;text-decoration:none">Open the site &nearr;</a>
  <div style="margin-top:12px;color:#4B4F49;font-size:14px;line-height:1.6">
    Recognition <b>whisper-small</b> &middot; speakers <b>{speakers}</b><br>
    The interface is the published site; the models run in this notebook.
  </div>
</div>
<iframe src="{link}" style="width:100%;height:760px;border:1px solid #C9C0A8"></iframe>
"""))

print("\nOPEN THIS:", link)
print("Backend  :", api)
print("Keep this notebook running for the link to stay alive.")